In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from football import add_form_features, poisson_walk_forward

columns = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "B365H", "B365D", "B365A"]

codes = []
for y in range(15, 26):
    codes.append(f"{y}{y + 1}")
print(codes)

seasons = []
for code in codes:
    s = pd.read_csv(f"https://www.football-data.co.uk/mmz4281/{code}/E0.csv")
    s = s[columns].dropna(subset=["HomeTeam"])
    s["Date"] = pd.to_datetime(s["Date"], dayfirst=True)
    s["Season"] = code
    seasons.append(s)

data = pd.concat(seasons).sort_values("Date").reset_index(drop=True)
print(len(data), "matches")

['1516', '1617', '1718', '1819', '1920', '2021', '2122', '2223', '2324', '2425', '2526']


/var/folders/56/t82_80m974df5kqxhh74dzsw0000gn/T/ipykernel_4269/756861354.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  s["Date"] = pd.to_datetime(s["Date"], dayfirst=True)


4180 matches


In [2]:
data = add_form_features(data)

parts = []
for i in range(1, len(codes)):
    old = data[data["Season"] == codes[i - 1]]
    new = data[data["Season"] == codes[i]]
    parts.append(poisson_walk_forward(old, new, 2))

pois = pd.concat(parts)
data = data.merge(pois, on=["Date", "HomeTeam", "AwayTeam"], how="left")

print(data.groupby("Season")["Date"].agg(["min", "max"]))
print(data.groupby("Season")["pois_H"].count())

              min        max
Season                      
1516   2015-08-08 2016-05-17
1617   2016-08-13 2017-05-21
1718   2017-08-11 2018-05-13
1819   2018-08-10 2019-05-12
1920   2019-08-09 2020-07-26
2021   2020-09-12 2021-05-23
2122   2021-08-13 2022-05-22
2223   2022-08-05 2023-05-28
2324   2023-08-11 2024-05-19
2425   2024-08-16 2025-05-25
2526   2025-08-15 2026-05-24
Season
1516      0
1617    380
1718    380
1819    380
1920    380
2021    380
2122    380
2223    380
2324    380
2425    380
2526    380
Name: pois_H, dtype: int64


In [3]:
def brier_on(results, pH, pD, pA):
    iH = (results == "H").astype(int)
    iD = (results == "D").astype(int)
    iA = (results == "A").astype(int)
    return ((pH - iH)**2 + (pD - iD)**2 + (pA - iA)**2).mean()

form = ["h_scored", "h_conceded", "a_scored", "a_conceded"]
combo = form + ["pois_H", "pois_A"]

rows = []
for i in range(2, len(codes)):
    tr = data[data["Season"].isin(codes[1:i])].dropna(subset=combo)
    te = data[data["Season"] == codes[i]].copy()
    te[form] = te[form].fillna(tr[form].mean())

    m_form = LogisticRegression(max_iter=1000).fit(tr[form], tr["FTR"])
    m_comb = LogisticRegression(max_iter=1000).fit(tr[combo], tr["FTR"])
    pf = m_form.predict_proba(te[form])
    pc = m_comb.predict_proba(te[combo])

    res = te["FTR"]
    mg = 1 / te["B365H"] + 1 / te["B365D"] + 1 / te["B365A"]
    b = tr["FTR"].value_counts(normalize=True)

    rows.append({
        "Season": codes[i],
        "Train": len(tr),
        "Baseline": brier_on(res, b["H"], b["D"], b["A"]),
        "Poisson": brier_on(res, te["pois_H"], te["pois_D"], te["pois_A"]),
        "Form": brier_on(res, pf[:, 2], pf[:, 1], pf[:, 0]),
        "Combined": brier_on(res, pc[:, 2], pc[:, 1], pc[:, 0]),
        "Bookmaker": brier_on(res, (1/te["B365H"])/mg, (1/te["B365D"])/mg, (1/te["B365A"])/mg),
    })

table = pd.DataFrame(rows)
print(table.round(4))
print()
print("Average over all seasons:")
print(table[["Baseline", "Poisson", "Form", "Combined", "Bookmaker"]].mean().round(4))

  Season  Train  Baseline  Poisson    Form  Combined  Bookmaker
0   1718    366    0.6474   0.5833  0.6073    0.5873     0.5600
1   1819    736    0.6304   0.5240  0.5772    0.5297     0.5221
2   1920   1101    0.6443   0.5903  0.6126    0.5898     0.5750
3   2021   1476    0.6649   0.6138  0.6479    0.6159     0.5982
4   2122   1851    0.6480   0.5833  0.6048    0.5799     0.5540
5   2223   2226    0.6337   0.6101  0.6140    0.5999     0.5735
6   2324   2601    0.6367   0.5654  0.5863    0.5594     0.5331
7   2425   2976    0.6563   0.5871  0.6274    0.5903     0.5787
8   2526   3351    0.6567   0.6367  0.6410    0.6292     0.6115

Average over all seasons:
Baseline     0.6465
Poisson      0.5882
Form         0.6132
Combined     0.5868
Bookmaker    0.5673
dtype: float64
